In [7]:
import json
import pandas as pd
import joblib

ensemble = joblib.load(
    r"E:\Work\AI\MAKTAB\HW-CW-S02\Mini_Project_01\models\fraud_ensemble.pkl"
)

scaler = joblib.load(
    r"E:\Work\AI\MAKTAB\HW-CW-S02\Mini_Project_01\models\scaler.pkl"
)

model_xg = ensemble["xgb"]
model_rf = ensemble["rf"]
knn_model = ensemble["knn"]

w_xgb = ensemble["w_xgb"]
w_rf = ensemble["w_rf"]
w_knn = ensemble["w_knn"]

threshold = ensemble["threshold"]

with open(
    r"E:\Work\AI\MAKTAB\HW-CW-S02\Mini_Project_01\data\predict_data.json",
    "r"
) as f:
    json_data = json.load(f)

data = pd.DataFrame([json_data])

feature_names = scaler.feature_names_in_

data = data[feature_names]

xgb_prob = model_xg.predict_proba(data)[:, 1]

rf_prob = model_rf.predict_proba(data)[:, 1]

data_scaled = scaler.transform(data)

knn_prob = knn_model.predict_proba(data_scaled)[:, 1]

ensemble_prob = (
    w_xgb * xgb_prob +
    w_rf * rf_prob +
    w_knn * knn_prob
)

probability = float(ensemble_prob[0])

class_id = int(probability >= threshold)

prediction = "Fraud" if class_id == 1 else "Normal"

result = {
    "prediction": prediction,
    "class_id": class_id,
    "probability": probability,
    "threshold": threshold,
    "status": "success"
}

with open(
    r"E:\Work\AI\MAKTAB\HW-CW-S02\Mini_Project_01\data\prediction_result.json",
    "w"
) as f:
    json.dump(result, f, indent=4)

print(json.dumps(result, indent=4))

{
    "prediction": "Normal",
    "class_id": 0,
    "probability": 8.674747200492141e-05,
    "threshold": 0.5600000000000002,
    "status": "success"
}


c:\Users\Amir\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
